In [1]:
!pip install requests beautifulsoup4 lxml fake-useragent tqdm

In [3]:
import requests
from bs4 import BeautifulSoup
from time import sleep
from tqdm import tqdm
import pandas as pd
import re

In [9]:
url = 'https://auto.drom.ru/audi/all/'
page = requests.get(url)
page

<Response [200]>

In [21]:
soup = BeautifulSoup(page.text, 'lxml') #тут смотрю структуру
print(soup.prettify())

<!DOCTYPE html>
<html class="drom-notouch" lang="ru" xml:lang="ru" xmlns="http://www.w3.org/1999/xhtml">
 <head>
  <title>
   Купить Ауди в России: продажа Audi с пробегом и новых от 17 700 рублей.
  </title>
  <meta content="IE=Edge" http-equiv="X-UA-Compatible"/>
  <meta content="drom.ru" name="copyright"/>
  <meta content="telephone=no" name="format-detection"/>
  <meta content="#000000" name="theme-color"/>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta charset="utf-8"/>
  <meta content='{"cf":{"p":{"min":"80000","max":"14875000"},"y":{"min":"1991","max":"2025"},"v":{"min":"1400","max":"4200"},"f":31,"category_id":1},"geor":77,"geoc":0,"id":27,"b":1,"bc":1,"charset":"utf-8"}' name="candy.config"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="13 828 объявлений о продаже Ауди б/у и новых в России от 17 700 рублей – частные объявления. Узнать стоимость Audi и купить с пробегом на Дроме" name="description"/>
  <meta content="ав

In [26]:
links = [] # ищу ссылки на объявления

for a in soup.find_all('a'):
    href = a.get('href')
    if href and re.match(r'https?://auto\.drom\.ru/.+/.+/.+/\d+\.html', href): # match показывает что ссылка подходит под наш паттерн
        links.append(href)

links = list(set(links)) #убираю дубликаты
links[:1000]

['https://auto.drom.ru/moscow/audi/a4/102813737.html',
 'https://auto.drom.ru/ekaterinburg/audi/a6/665811532.html',
 'https://auto.drom.ru/moscow/audi/q5/596509490.html',
 'https://auto.drom.ru/moscow/audi/a8/732297298.html',
 'https://auto.drom.ru/moscow/audi/a7/710480443.html',
 'https://auto.drom.ru/ekaterinburg/audi/q3/470145595.html',
 'https://auto.drom.ru/moscow/audi/a1/574598269.html',
 'https://auto.drom.ru/moscow/audi/a6/217602654.html',
 'https://auto.drom.ru/habarovsk/audi/a3/738096062.html',
 'https://auto.drom.ru/moscow/audi/q5/332859885.html',
 'https://auto.drom.ru/vidnoe/audi/q5/179337071.html',
 'https://auto.drom.ru/moscow/audi/q7/549980325.html',
 'https://auto.drom.ru/moscow/audi/q7/758275176.html',
 'https://auto.drom.ru/spb/audi/q5/405910569.html',
 'https://auto.drom.ru/moscow/audi/a4/162672695.html',
 'https://auto.drom.ru/moscow/audi/s4/480600179.html',
 'https://auto.drom.ru/izhevsk/audi/q3/517369960.html',
 'https://auto.drom.ru/ufa/audi/q3/738450225.html',


In [5]:
from fake_useragent import UserAgent
fake_user = UserAgent()
headers = {'User-Agent': fake_user.random}

In [54]:
url0 = links[1]
page0 = requests.get(url0, headers = headers) #это чтобы рандомные параметры браузера были,чтобы не блочило
soup0 = BeautifulSoup(page0.text, 'lxml') #беру первое объявление и смотрю что внутри
text = soup0.get_text(separator='\n', strip = True) #это чтобы с новой строки были данные и стрип, чтобы пробелов лишних не было, а то там проблемы огромные
print(text[:3000])

Продажа Ауди А6 15 года в Екатеринбурге, Продам Ауди а6 2015г, обмен на более дорогую, комплектация 2.8 FSI quattro S tronic Business, коробка AT, полный привод
Продать
Автомобили
Запчасти
Продажа шин
Отзывы
Каталог
Аукционы Японии
Автомобили из Китая
Автомобили из Кореи
Автомобили из Германии
Электромобили
Каталог китайских авто
ОСАГО онлайн
Автокредиты
Проверка по VIN
Оценить автомобиль
Форумы
ПДД онлайн
Вопросы и ответы
Рейтинг авто
Каталог шин
Договор купли-продажи
Правовые вопросы
Карта сайта
Размещение на Дроме
Разместить прайс
Помощь
Сделка с Дромом на 3 000 000 ₽
Дром
Продажа авто в Екатеринбурге
Audi
A6
Объявление 665811532
Audi A6 2015 в Екатеринбурге
1
/
18
2 200 000
₽
высокая цена
В
кредит
от
39 353
₽
в месяц
Двигатель
бензин, 2.8 л
Мощность
220
л.с.
,
налог
Коробка передач
робот
Привод
4WD
Тип кузова
седан
Цвет
белый
Пробег
238 300
км
Владельцы
3
Руль
левый
Поколение
4 поколение, рестайлинг
Комплектация
2.8 FSI quattro S tronic Business
Отчет по VIN-коду
WAU**************


In [9]:
def GetCar(url0):
    page0 = requests.get(url0, headers = headers)
    soup0 = BeautifulSoup(page0.text, 'lxml')
    car = {}
    car['url'] = url0
    try:
        car['title'] = soup0.find('h1').get_text(strip=True)
    except AttributeError:
        car['title'] = None
    if car['title']:
        year = re.findall(r'\d{4}', car['title'])
        if year:
            car['year'] = int(year[0]) #беру год из заголовка

    #тут марка, модель и город из ссылки
    parts = url0.split('/')
    car['city'] = parts[3]
    car['make'] = parts[4]
    car['model'] = parts[5]

    #дальше работаю с текстом страницы построчно
    text = soup0.get_text('\n', strip=True)
    lines = text.split('\n')

    car['price'] = None
    for line in lines:
        s = line.replace(' ', '').replace('\xa0', '')
        if s.isdigit():
            if 50000 < int(s) < 100000000:
                car['price'] = int(s)
                break

    #еще характеристики
    car['engine'] = None
    car['transmission'] = None
    car['mileage'] = None

    for i in range(len(lines) - 1):
        key = lines[i].lower().strip()
        value = lines[i+1].strip() #тут инфа о параметре на строчку ниже его
        if key == 'двигатель':
            car['engine'] = value
        if key == 'коробка передач':
            car['transmission'] = value
        if key == 'пробег':
            km = re.findall(r'\d+', value.replace(' ', '').replace('\xa0', ''))
            if km:
                car['mileage'] = int(km[0])
    return car

In [11]:
test = GetCar('https://auto.drom.ru/moscow/audi/a4/162672695.html')
test

{'url': 'https://auto.drom.ru/moscow/audi/a4/162672695.html',
 'title': 'Продажа Audi A4, 2001 год в Москве',
 'year': 2001,
 'city': 'moscow',
 'make': 'audi',
 'model': 'a4',
 'price': 850000,
 'engine': 'бензин, 2.0 л',
 'transmission': 'вариатор',
 'mileage': 177000}

In [13]:
def GetCar(url0):
    page0 = requests.get(url0, headers = headers)
    soup0 = BeautifulSoup(page0.text, 'lxml')
    car = {}
    car['url'] = url0
    try:
        car['title'] = soup0.find('h1').get_text(strip=True)
    except AttributeError:
        car['title'] = None
    if car['title']:
        year = re.findall(r'\d{4}', car['title'])
        if year:
            car['year'] = int(year[0])

    #тут марка, модель и город из ссылки
    parts = url0.split('/')
    car['city'] = parts[3]
    car['make'] = parts[4]
    car['model'] = parts[5]

    #дальше работаю с текстом страницы построчно
    text = soup0.get_text('\n', strip=True)
    lines = text.split('\n')

    car['price'] = None
    for line in lines:
        s = line.replace(' ', '').replace('\xa0', '')
        if s.isdigit():
            if 50000 < int(s) < 100000000:
                car['price'] = int(s)
                break

    #еще характеристики
    car['engine'] = None
    car['transmission'] = None
    car['mileage'] = None
    car['drive'] = None
    car['body'] = None
    car['color'] = None
    car['wheel'] = None
    car['hp'] = None

    for i in range(len(lines) - 1):
        key = lines[i].lower().strip()
        value = lines[i+1].strip()
        if key == 'двигатель':
            car['engine'] = value
        if key == 'мощность':
            car['hp'] = int(value)
        if key == 'коробка передач':
            car['transmission'] = value
        if key == 'привод':
            car['drive'] = value
        if key == 'тип кузова':
            car['body'] = value
        if key == 'цвет':
            car['color'] = value
        if key == 'руль':
            car['wheel'] = value
        if key == 'пробег':
            km = re.findall(r'\d+', value.replace(' ', '').replace('\xa0', ''))
            if km:
                car['mileage'] = int(km[0])

    #описание продавца
    car['description'] = None
    for i in range(len(lines)):
        if 'дополнительно' in lines[i].lower():
            car['description'] = ' '.join(lines[i+1:i+20])
            break

    return car

In [15]:
test = GetCar('https://auto.drom.ru/habarovsk/toyota/land_cruiser_prado/921709892.html')
test

{'url': 'https://auto.drom.ru/habarovsk/toyota/land_cruiser_prado/921709892.html',
 'title': 'Продажа Toyota Land Cruiser Prado, 2001 год в Хабаровске',
 'year': 2001,
 'city': 'habarovsk',
 'make': 'toyota',
 'model': 'land_cruiser_prado',
 'price': 2575000,
 'engine': 'бензин, 3.4 л',
 'transmission': 'АКПП',
 'mileage': 156700,
 'drive': '4WD',
 'body': 'джип/suv 5 дв.',
 'color': 'зеленый',
 'wheel': 'правый',
 'hp': 185,
 'description': ': Без пробега по России в наличии в Хабаровске ! Оформлен! Птс на руках бумажный я собственник! Продаю Привезли из Японии с юга, поэтому на джипе нет ни гнили ни ржавчины! Рама состояние новой! Бензин 3.4 объём , мотор 5vz-fe, простой без какой либо электроники! Люк! Вторая печка салона, третий ряд сидений! Высокий бардачек! Весь в родной краске! Ничего не красилось не мазалось! Все до болтика на машине из Японии, без пробега по России! Все наклейки о заменах жидкостей все приклеены на своих местах! Не бутафория !!!! Состояние нового джипа! Салон 

In [13]:
def get_links(make, p):
    url = f'https://auto.drom.ru/{make}/all/page{p}/'
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'lxml')
    
    links = []
    for a in soup.find_all('a'):
        href = a.get('href')
        if href and re.match(r'https?://auto\.drom\.ru/.+/.+/.+/\d+\.html', href):
            links.append(href)
    
    return list(set(links))

In [88]:
test_links = get_links('toyota', 1)
print(len(test_links))
test_links[:5]

26


['https://auto.drom.ru/moscow/toyota/sienta/239292244.html',
 'https://auto.drom.ru/moscow/toyota/highlander/336808218.html',
 'https://auto.drom.ru/moscow/toyota/tacoma/173302475.html',
 'https://auto.drom.ru/moscow/toyota/land_cruiser/390458277.html',
 'https://auto.drom.ru/moscow/toyota/tundra/288523432.html']

In [39]:
makes_batch = ['toyota', 'hyundai', 'kia']
links_1 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_1.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_1))}).to_csv('batch_1.csv', index=False)
print(f'пачка 1: {len(set(links_1))}')

собираю toyota


100%|███████████████████████████████████████████| 15/15 [01:30<00:00,  6.03s/it]


собираю hyundai


100%|███████████████████████████████████████████| 15/15 [01:25<00:00,  5.70s/it]


собираю kia


100%|███████████████████████████████████████████| 15/15 [01:32<00:00,  6.16s/it]

пачка 1: 915


In [41]:
makes_batch = ['volkswagen', 'nissan', 'honda']
links_2 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_2.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_2))}).to_csv('batch_2.csv', index=False)
print(f'пачка 2: {len(set(links_2))}')

собираю volkswagen


100%|███████████████████████████████████████████| 15/15 [01:31<00:00,  6.10s/it]


собираю nissan


100%|███████████████████████████████████████████| 15/15 [01:30<00:00,  6.03s/it]


собираю honda


100%|███████████████████████████████████████████| 15/15 [01:29<00:00,  5.98s/it]

пачка 2: 639


In [42]:
makes_batch = ['mazda', 'bmw', 'ford']
links_3 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_3.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_3))}).to_csv('batch_3.csv', index=False)
print(f'пачка 3: {len(set(links_3))}')

собираю mazda


100%|███████████████████████████████████████████| 15/15 [01:20<00:00,  5.34s/it]


собираю bmw


100%|███████████████████████████████████████████| 15/15 [01:27<00:00,  5.84s/it]


собираю ford


100%|███████████████████████████████████████████| 15/15 [01:28<00:00,  5.91s/it]

пачка 3: 0


In [43]:
makes_batch = ['lada', 'mitsubishi', 'renault']
links_4 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_4.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_4))}).to_csv('batch_4.csv', index=False)
print(f'пачка 4: {len(set(links_4))}')

собираю lada


100%|███████████████████████████████████████████| 15/15 [01:26<00:00,  5.74s/it]


собираю mitsubishi


100%|███████████████████████████████████████████| 15/15 [01:22<00:00,  5.53s/it]


собираю renault


100%|███████████████████████████████████████████| 15/15 [01:23<00:00,  5.57s/it]

пачка 4: 0


In [44]:
makes_batch = ['skoda', 'haval', 'chery']
links_5 = []

for make in makes_batch:
    print(f'собираю {make}')
    for p in tqdm(range(1, 16)):
        try:
            links_5.extend(get_links(make, p))
            sleep(4)
        except:
            print(f'ошибка {make} {p}')

pd.DataFrame({'url': list(set(links_5))}).to_csv('batch_5.csv', index=False)
print(f'пачка 5: {len(set(links_5))}')

собираю skoda


100%|███████████████████████████████████████████| 15/15 [01:21<00:00,  5.45s/it]


собираю haval


100%|███████████████████████████████████████████| 15/15 [01:21<00:00,  5.44s/it]


собираю chery


100%|███████████████████████████████████████████| 15/15 [01:20<00:00,  5.34s/it]

пачка 5: 0


In [114]:
b1 = pd.read_csv('batch_1.csv')
b2 = pd.read_csv('batch_2.csv')
b3 = pd.read_csv('batch_3.csv')
b4 = pd.read_csv('batch_4.csv')
b5 = pd.read_csv('batch_5.csv')
all_df = pd.concat([b1, b2, b3, b4, b5])
all_links = all_df['url'].tolist()
pd.DataFrame({'url': all_links}).to_csv('all_links.csv', index=False)

In [116]:
all_links = list(set(all_links))
print(f'собрано: {len(all_links)}')

собрано: 4584


In [31]:
cars = []

for i, link in enumerate(tqdm(all_links)):
    try:
        car = GetCar(link)
        cars.append(car)
        sleep(3)
    except:
        print(link)
    if (i+1) % 200 == 0:  #сохраняю каждые 200 на всякий случай
        df = pd.DataFrame(cars)
        df.to_csv(f'checkpoint_{i+1}.csv', index=False)

100%|█████████████████████████████████████| 6332/6332 [5:42:35<00:00,  3.25s/it]


In [94]:
df = pd.DataFrame(cars)
print(df.shape)

NameError: name 'cars' is not defined

In [19]:
df_links = pd.read_csv('all_links_6_13.csv')
all_links = df_links['url'].tolist()

In [21]:
links_1 = all_links[0:1266]
links_2 = all_links[1266:2532]
links_3 = all_links[2532:3798]
links_4 = all_links[3798:5064]
links_5 = all_links[5064:6332]

In [33]:
pd.DataFrame({'url': links_1}).to_csv('links_1.csv', index=False)
pd.DataFrame({'url': links_2}).to_csv('links_2.csv', index=False)
pd.DataFrame({'url': links_3}).to_csv('links_3.csv', index=False)
pd.DataFrame({'url': links_4}).to_csv('links_4.csv', index=False)
pd.DataFrame({'url': links_5}).to_csv('links_5.csv', index=False)

In [23]:
len(links_4)

1266

In [25]:
import random

In [39]:
links_1_cars = []

for i, link in enumerate(tqdm(links_1)):
    try:
        car = GetCar(link)
        links_1_cars.append(car)
        sleep(random.uniform(1, 7))
    except:
        print(link)
    if (i+1) % 50 == 0:  #сохраняю каждые 50 на всякий случай
        df_links_1 = pd.DataFrame(links_1_cars)
        df_links_1.to_csv(f'links_1_checkpoint_{i+1}.csv', index=False)

  4%|███▍                                                                             | 53/1266 [05:58<2:07:14,  6.29s/it]

https://auto.drom.ru/moscow/audi/q8/439320272.html


100%|███████████████████████████████████████████████████████████████████████████████| 1266/1266 [1:37:41<00:00,  4.63s/it]


In [43]:
links_1_total = pd.DataFrame(links_1_cars)
links_1_total.to_csv(f'links_1_total.csv', index=False)

In [27]:
links_2_5_cars = []

for i, link in enumerate(tqdm(all_links[1266:])):
    try:
        car = GetCar(link)
        links_2_5_cars.append(car)
        sleep(random.uniform(3, 7))
    except:
        print(link)
    if (i+1) % 30 == 0:  #сохраняю каждые 50 на всякий случай
        sleep(random.uniform(20, 30))
    if (i+1) % 50 == 0:  #сохраняю каждые 50 на всякий случай
        df_links_2_5 = pd.DataFrame(links_2_5_cars)
        df_links_2_5.to_csv(f'links_2_5_checkpoint_{i+1}.csv', index=False)

  8%|██▋                              | 421/5066 [1:17:21<787:59:45, 610.72s/it]

https://auto.drom.ru/yevpatoriya/fiat/palio/674380166.html
https://auto.drom.ru/mihaylovka-irk/fiat/albea/185243045.html
https://auto.drom.ru/kaliningrad/citroen/c5/883396703.html
https://auto.drom.ru/nyagan/opel/insignia/804181363.html
https://auto.drom.ru/solikamsk/fiat/linea/126632692.html
https://auto.drom.ru/orsha_rb/opel/insignia/796227986.html
https://auto.drom.ru/penza/citroen/c4/149979036.html
https://auto.drom.ru/sevastopol/fiat/linea/665442724.html
https://auto.drom.ru/spb/citroen/berlingo/926654552.html
https://auto.drom.ru/spb/opel/astra/944914205.html
https://auto.drom.ru/ekaterinburg/fiat/punto/881146845.html
https://auto.drom.ru/kemerovo/opel/vectra/606397160.html
https://auto.drom.ru/spb/opel/astra/623531755.html
https://auto.drom.ru/tver/citroen/c-crosser/888665323.html
https://auto.drom.ru/izhevsk/opel/astra/203990508.html
https://auto.drom.ru/tyumen/fiat/doblo/762577629.html
https://auto.drom.ru/omutinskoe/fiat/doblo/510690923.html
https://auto.drom.ru/ufa/opel/anta

  9%|██▉                              | 460/5066 [1:56:13<201:50:09, 157.75s/it]

https://auto.drom.ru/salavat/opel/insignia/730818479.html
https://auto.drom.ru/omsk/fiat/albea/269992393.html
https://auto.drom.ru/minsk/citroen/c5_aircross/909889702.html
https://auto.drom.ru/podolsk/fiat/doblo/235900433.html
https://auto.drom.ru/voronezh/citroen/c4/571379098.html
https://auto.drom.ru/simferopol/opel/astra/857461650.html
https://auto.drom.ru/perm/opel/corsa/826198988.html
https://auto.drom.ru/tomsk/opel/vectra/221345741.html
https://auto.drom.ru/kropotkin/citroen/c4/231483969.html
https://auto.drom.ru/ekaterinburg/opel/antara/876280752.html
https://auto.drom.ru/moscow/citroen/c4/119233318.html
https://auto.drom.ru/naberezhnye-chelny/fiat/albea/637400125.html
https://auto.drom.ru/barnaul/opel/astra/482561229.html
https://auto.drom.ru/yalta/opel/corsa/833306696.html
https://auto.drom.ru/vladikavkaz/citroen/ds4/208638066.html
https://auto.drom.ru/fokino-bryansk/citroen/c4/398961194.html
https://auto.drom.ru/kurgan/fiat/punto/159025583.html
https://auto.drom.ru/naberezhny

  9%|███▎                               | 479/5066 [1:56:28<18:35:26, 14.59s/it]


KeyboardInterrupt: 

In [22]:
links_2_5_total = pd.DataFrame(links_2_5_cars)
links_2_5_total.to_csv(f'links_2_5_total.csv', index=False)

In [29]:
len(links_2)

1266

In [ ]:
links_2_cars = []

for i, link in enumerate(tqdm(links_2)):
    try:
        car = GetCar(link)
        links_2_cars.append(car)
        sleep(random.uniform(1, 7))
    except:
        print(link)
    if (i+1) % 50 == 0:  #сохраняю каждые 50 на всякий случай
        df_links_2 = pd.DataFrame(links_2_cars)
        df_links_2.to_csv(f'links_2_checkpoint_{i+1}.csv', index=False)

 27%|██████████▏                           | 339/1266 [32:31<1:13:31,  4.76s/it]

https://auto.drom.ru/tyumen/opel/zafira/903577906.html


In [24]:
links_3_cars = []

for i, link in enumerate(tqdm(links_3)):
    try:
        car = GetCar(link)
        links_3_cars.append(car)
        sleep(random.uniform(1, 7))
    except:
        print(link)
    if (i+1) % 50 == 0:  #сохраняю каждые 50 на всякий случай
        df_links_3 = pd.DataFrame(links_3_cars)
        df_links_3.to_csv(f'links_3_checkpoint_{i+1}.csv', index=False)

  0%|                                                  | 0/1266 [00:00<?, ?it/s]

https://auto.drom.ru/kursk/chevrolet/lacetti/349148176.html


 23%|████████▊                             | 292/1266 [00:00<00:00, 1463.48it/s]

https://auto.drom.ru/moscow/porsche/cayenne/933726063.html
https://auto.drom.ru/dmitriev/chevrolet/lacetti/293827639.html
https://auto.drom.ru/omsk/chevrolet/niva/806650406.html
https://auto.drom.ru/novosibirsk/porsche/cayenne/51789268.html
https://auto.drom.ru/omsk/porsche/cayenne/162685461.html
https://auto.drom.ru/moscow/porsche/taycan/463740398.html
https://auto.drom.ru/vladivostok/porsche/cayenne/744216133.html
https://auto.drom.ru/irkutsk/chevrolet/cobalt/626413419.html
https://auto.drom.ru/chelyabinsk/chevrolet/lacetti/343487899.html
https://auto.drom.ru/sochi/mini/hatch/299705431.html
https://auto.drom.ru/moscow/mini/hatch/218563860.html
https://auto.drom.ru/spb/mini/hatch/133062183.html
https://auto.drom.ru/novosibirsk/chevrolet/cruze/837385662.html
https://auto.drom.ru/habarovsk/mini/countryman/741213217.html
https://auto.drom.ru/barnaul/chevrolet/niva/675982193.html
https://auto.drom.ru/nizhnevartovsk/porsche/cayenne/498860805.html
https://auto.drom.ru/sterlitamak/chevrolet/

 49%|██████████████████▍                   | 616/1266 [00:00<00:00, 1579.17it/s]

...

https://auto.drom.ru/moscow/mini/hatch/547454337.html
https://auto.drom.ru/moscow/porsche/macan/138132379.html
https://auto.drom.ru/nizhniy-novgorod/mini/hatch/362860136.html
https://auto.drom.ru/moscow/porsche/cayenne/819341524.html
https://auto.drom.ru/kemerovo/mini/hatch/654059078.html
https://auto.drom.ru/krasnodar/mini/hatch/178258821.html
https://auto.drom.ru/moscow/porsche/911/911679633.html
https://auto.drom.ru/moscow/porsche/cayenne_coupe/782850212.html
https://auto.drom.ru/moscow/porsche/macan/609379323.html
https://auto.drom.ru/ekaterinburg/chevrolet/lacetti/156407654.html
https://auto.drom.ru/krasnodar/porsche/cayenne/786785475.html
https://auto.drom.ru/tyumen/porsche/cayenne/944441915.html
https://auto.drom.ru/moscow/porsche/cayenne/827021925.html
https://auto.drom.ru/novosibirsk/chevrolet/niva/480421237.html
https://auto.drom.ru/tambov/chevrolet/niva/386933642.html
https://auto.drom.ru/kurgan/mini/hatch/210062360.html
https://auto.drom.ru/simferopol/chevrolet/lacetti/3864

 92%|████████████████████████████████████   | 1169/1266 [17:55<03:01,  1.87s/it]

https://auto.drom.ru/yessentuki/tesla/model_3/632754356.html
https://auto.drom.ru/ekaterinburg/cadillac/srx/598624147.html
https://auto.drom.ru/novosibirsk/cadillac/escalade/870270818.html
https://auto.drom.ru/vladivostok/jeep/wrangler/674639341.html
https://auto.drom.ru/omsk/jeep/commander/822062490.html
https://auto.drom.ru/togliatti/jeep/grand_cherokee/668716153.html
https://auto.drom.ru/moscow/jeep/wagoneer/884435575.html
https://auto.drom.ru/vladivostok/jeep/wrangler/791061064.html
https://auto.drom.ru/novorossiysk/tesla/model_3/567662164.html
https://auto.drom.ru/chita/jeep/wrangler/875939010.html
https://auto.drom.ru/yessentuki/tesla/model_3/655940814.html
https://auto.drom.ru/ulyanovsk/cadillac/cts/861738324.html
https://auto.drom.ru/kaliningrad/tesla/model_x/811813584.html
https://auto.drom.ru/habarovsk/jeep/wrangler/304165808.html
https://auto.drom.ru/spb/jeep/wrangler/508977755.html
https://auto.drom.ru/novosibirsk/tesla/model_s/176153667.html
https://auto.drom.ru/berezniky/

100%|███████████████████████████████████████| 1266/1266 [17:55<00:00,  1.18it/s]

https://auto.drom.ru/omsk/jeep/grand_cherokee/532936484.html
https://auto.drom.ru/kaluga/jeep/grand_cherokee/178889174.html
https://auto.drom.ru/gomel/tesla/model_y/656338478.html
https://auto.drom.ru/novokuznetsk/tesla/model_3/786209053.html
https://auto.drom.ru/moscow/cadillac/escalade/415823980.html
https://auto.drom.ru/vladikavkaz/jeep/grand_cherokee/789060562.html
https://auto.drom.ru/noyabrsk/jeep/grand_cherokee/498104683.html
https://auto.drom.ru/kostroma/jeep/wrangler/150038385.html
https://auto.drom.ru/simferopol/tesla/model_s/102126221.html
https://auto.drom.ru/vladivostok/cadillac/escalade/816042792.html
https://auto.drom.ru/noviy-urengoy/cadillac/escalade/678686694.html
https://auto.drom.ru/moscow/cadillac/xt5/467922118.html
https://auto.drom.ru/spb/jeep/grand_cherokee/856961409.html
https://auto.drom.ru/chelyabinsk/tesla/model_3/690057634.html
https://auto.drom.ru/spb/cadillac/srx/484113633.html
https://auto.drom.ru/feodosiya/jeep/grand_cherokee/341278339.html
https://auto

In [25]:
links_4_cars = []

for i, link in enumerate(tqdm(links_4)):
    try:
        car = GetCar(link)
        links_4_cars.append(car)
        sleep(random.uniform(1, 7))
    except:
        print(link)
    if (i+1) % 50 == 0:  #сохраняю каждые 50 на всякий случай
        df_links_4 = pd.DataFrame(links_4_cars)
        df_links_4.to_csv(f'links_4_checkpoint_{i+1}.csv', index=False)

  9%|███▌                                  | 119/1266 [00:00<00:00, 1189.03it/s]

https://auto.drom.ru/shchelkovo/cadillac/cts/939155675.html
https://auto.drom.ru/spb/tesla/model_3/708479453.html
https://auto.drom.ru/blagoveshchensk/cadillac/escalade/48412784.html
https://auto.drom.ru/krasnoyarsk/cadillac/bls/671392412.html
https://auto.drom.ru/velikiy-novgorod/cadillac/ats/473673400.html
https://auto.drom.ru/minsk/tesla/model_s/337464119.html
https://auto.drom.ru/moscow/jeep/compass/802299783.html
https://auto.drom.ru/chita/cadillac/cts/307831613.html
https://auto.drom.ru/apsheronsk/tesla/model_s/866122390.html
https://auto.drom.ru/yuzhno-sakhalinsk/tesla/model_y/520061504.html
https://auto.drom.ru/pyatigorsk/cadillac/escalade/152153392.html
https://auto.drom.ru/spb/jeep/grand_cherokee/306100799.html
https://auto.drom.ru/vladivostok/jeep/wrangler/382987246.html
https://auto.drom.ru/sochi/cadillac/srx/562586709.html
https://auto.drom.ru/krasnodar/jeep/wrangler/764097859.html
https://auto.drom.ru/vladivostok/cadillac/xt5/921017893.html
https://auto.drom.ru/novosibirs

 22%|████████▍                             | 283/1266 [00:00<00:00, 1450.34it/s]

https://auto.drom.ru/surgut/cadillac/escalade/199053711.html
https://auto.drom.ru/moscow/jeep/grand_cherokee/133760510.html
https://auto.drom.ru/sochi/jeep/wrangler/422813392.html
https://auto.drom.ru/spb/tesla/model_3/644395330.html
https://auto.drom.ru/volokolamsk/tesla/model_y/724216819.html
https://auto.drom.ru/moscow/cadillac/escalade/735488855.html
https://auto.drom.ru/moscow/tesla/model_3/539326260.html
https://auto.drom.ru/yuzhno-sakhalinsk/tesla/model_3/53749457.html
https://auto.drom.ru/nizhniy-novgorod/cadillac/escalade/159080158.html
https://auto.drom.ru/moscow/cadillac/escalade/504079984.html
https://auto.drom.ru/tyumen/tesla/model_3/709916005.html
https://auto.drom.ru/habarovsk/tesla/model_3/346143602.html
https://auto.drom.ru/moscow/jeep/compass/834931057.html
https://auto.drom.ru/irkutsk/cadillac/cts/433247219.html
https://auto.drom.ru/rostov-na-donu/jeep/wrangler/305477178.html
https://auto.drom.ru/samara/cadillac/escalade/769786069.html
https://auto.drom.ru/donetsk/ca

 34%|█████████████                         | 435/1266 [00:00<00:00, 1478.98it/s]

...

https://auto.drom.ru/moscow/dodge/caravan/513430584.html
https://auto.drom.ru/ufa/subaru/forester/398296834.html
https://auto.drom.ru/nakhodka/dodge/ram/632476932.html
https://auto.drom.ru/novopavlovsk/dodge/caravan/752153450.html
https://auto.drom.ru/vladivostok/subaru/forester/609618157.html
https://auto.drom.ru/barnaul/subaru/legacy_b4/913449169.html
https://auto.drom.ru/blagoveshchensk/subaru/levorg/871420014.html
https://auto.drom.ru/oktyabrsk/dodge/challenger/458646726.html
https://auto.drom.ru/moscow/dodge/challenger/408987410.html
https://auto.drom.ru/moscow/dodge/challenger/380425100.html
https://auto.drom.ru/kazan/suzuki/sx4/295454810.html
https://auto.drom.ru/izhevsk/dodge/charger/501840411.html
https://auto.drom.ru/novorossiysk/subaru/forester/602096020.html
https://auto.drom.ru/kaluga/suzuki/grand_vitara/639241922.html
https://auto.drom.ru/moscow/subaru/legacy/552106818.html
https://auto.drom.ru/odintsovo/dodge/caliber/937427614.html
https://auto.drom.ru/kursk/dodge/grand_

 86%|███████████████████████████████▉     | 1092/1266 [00:00<00:00, 1582.84it/s]

https://auto.drom.ru/novosibirsk/dodge/ram/368455339.html
https://auto.drom.ru/irkutsk/suzuki/kei/254908720.html
https://auto.drom.ru/habarovsk/suzuki/swift/343832519.html
https://auto.drom.ru/habarovsk/suzuki/swift/799682872.html
https://auto.drom.ru/novosibirsk/dodge/caravan/157844680.html
https://auto.drom.ru/vladivostok/suzuki/swift/670956006.html
https://auto.drom.ru/habarovsk/suzuki/swift/783848070.html
https://auto.drom.ru/moscow/dodge/challenger/53406255.html
https://auto.drom.ru/kirov/subaru/forester/217614504.html
https://auto.drom.ru/vladivostok/suzuki/every/387569569.html
https://auto.drom.ru/pskov/dodge/caravan/162754562.html
https://auto.drom.ru/engels/dodge/challenger/734661377.html
https://auto.drom.ru/kemerovo/dodge/neon/108287843.html
https://auto.drom.ru/simferopol/suzuki/sx4/788474994.html
https://auto.drom.ru/petropavlovsk-kamchatskiy/suzuki/escudo/129873507.html
https://auto.drom.ru/vladivostok/suzuki/jimny/160250583.html
https://auto.drom.ru/angarsk/suzuki/cultus

100%|█████████████████████████████████████| 1266/1266 [00:00<00:00, 1522.70it/s]

https://auto.drom.ru/vladivostok/suzuki/solio/209980752.html
https://auto.drom.ru/omsk/dodge/durango/152671789.html
https://auto.drom.ru/vladivostok/subaru/forester/809841224.html
https://auto.drom.ru/krasnodar/dodge/dakota/768908205.html
https://auto.drom.ru/omsk/dodge/caravan/103552769.html
https://auto.drom.ru/desnogorsk/dodge/neon/804572223.html
https://auto.drom.ru/grozniy/dodge/grand_caravan/807888831.html
https://auto.drom.ru/spb/dodge/challenger/583282336.html
https://auto.drom.ru/lyubertsy/suzuki/samurai/828941199.html
https://auto.drom.ru/moscow/dodge/grand_caravan/778999012.html
https://auto.drom.ru/simferopol/suzuki/swift/433156930.html
https://auto.drom.ru/vladivostok/suzuki/jimny/914364971.html
https://auto.drom.ru/beslan/suzuki/solio/763512048.html
https://auto.drom.ru/petrozavodsk/suzuki/sx4/415707037.html
https://auto.drom.ru/yuzhno-sakhalinsk/subaru/forester/206054143.html
https://auto.drom.ru/magadan/suzuki/swift/599334144.html
https://auto.drom.ru/shushenskoe/dodge/

In [26]:
links_5_cars = []

for i, link in enumerate(tqdm(links_5)):
    try:
        car = GetCar(link)
        links_5_cars.append(car)
        sleep(random.uniform(1, 7))
    except:
        print(link)
    if (i+1) % 50 == 0:  #сохраняю каждые 50 на всякий случай
        df_links_5 = pd.DataFrame(links_5_cars)
        df_links_5.to_csv(f'links_5_checkpoint_{i+1}.csv', index=False)

 11%|████▎                                 | 142/1268 [00:00<00:00, 1414.64it/s]

https://auto.drom.ru/zaozersk/dodge/journey/816490138.html
https://auto.drom.ru/lipovtsy/suzuki/wagon_r_plus/616349363.html
https://auto.drom.ru/maloyaroslavets/subaru/outback/630418094.html
https://auto.drom.ru/petrozavodsk/suzuki/grand_vitara/111690973.html
https://auto.drom.ru/petropavlovsk-kamchatskiy/dodge/caliber/590495028.html
https://auto.drom.ru/biysk/subaru/forester/693157720.html
https://auto.drom.ru/abakan/suzuki/jimny/877952690.html
https://auto.drom.ru/dalnegorsk/subaru/forester/257508322.html
https://auto.drom.ru/sochi/subaru/forester/220779271.html
https://auto.drom.ru/velikiy-novgorod/suzuki/grand_vitara/662480635.html
https://auto.drom.ru/ulan-ude/suzuki/grand_vitara/536131831.html
https://auto.drom.ru/ekaterinburg/dodge/caravan/483197505.html
https://auto.drom.ru/habarovsk/subaru/forester/267935036.html
https://auto.drom.ru/oktyabrsk/dodge/caliber/528665832.html
https://auto.drom.ru/spb/dodge/stratus/893297694.html
https://auto.drom.ru/krasnoyarsk/subaru/levorg/54504

 33%|████████████▌                         | 420/1268 [00:00<00:00, 1327.31it/s]

https://auto.drom.ru/krasnoyarsk/zeekr/7x/480452966.html
https://auto.drom.ru/iskitim/infiniti/fx35/447995303.html
https://auto.drom.ru/irkutsk/lexus/ct200h/116055239.html
https://auto.drom.ru/ekaterinburg/zeekr/001/291703657.html
https://auto.drom.ru/omsk/infiniti/m35/341117125.html
https://auto.drom.ru/ufa/zeekr/x/608799249.html
https://auto.drom.ru/zaozerniy/infiniti/fx35/535119002.html
https://auto.drom.ru/yakutsk/lexus/gx470/575507811.html
https://auto.drom.ru/blagoveshchensk/infiniti/qx80/253536506.html
https://auto.drom.ru/moscow/lexus/rx500h/417411134.html
https://auto.drom.ru/novosibirsk/infiniti/ex35/117419522.html
https://auto.drom.ru/moscow/zeekr/7x/263730496.html
https://auto.drom.ru/mogocha/lexus/rx330/860049226.html
https://auto.drom.ru/novosibirsk/lexus/gx470/866001739.html
https://auto.drom.ru/omsk/infiniti/fx35/493870120.html
https://auto.drom.ru/moscow/lexus/es250/594150622.html
https://auto.drom.ru/moscow/zeekr/9x/362212790.html
https://auto.drom.ru/moscow/infiniti/

 55%|████████████████████▉                 | 699/1268 [00:00<00:00, 1363.31it/s]

https://auto.drom.ru/novokuznetsk/lexus/rx300/718241346.html
https://auto.drom.ru/moscow/zeekr/9x/461019518.html
https://auto.drom.ru/nizhnevartovsk/infiniti/qx70/271447622.html
https://auto.drom.ru/pushkino/lexus/gs250/766611164.html
https://auto.drom.ru/barnaul/lexus/rx350/459276468.html
https://auto.drom.ru/novosibirsk/lexus/rx300/546831488.html
https://auto.drom.ru/omsk/infiniti/fx35/678629265.html
https://auto.drom.ru/ussuriisk/lexus/hs250h/949268202.html
https://auto.drom.ru/moscow/zeekr/9x/338352297.html
https://auto.drom.ru/moscow/zeekr/9x/537186434.html
https://auto.drom.ru/tomsk/infiniti/qx70/50302482.html
https://auto.drom.ru/moscow/zeekr/9x/169114163.html
https://auto.drom.ru/bratsk/infiniti/qx56/776323432.html
https://auto.drom.ru/novosibirsk/infiniti/fx35/588866841.html
https://auto.drom.ru/barnaul/lexus/lx570/469824044.html
https://auto.drom.ru/moscow/zeekr/001/216271849.html
https://auto.drom.ru/raduzhniy-hanty/lexus/lx470/753610637.html
https://auto.drom.ru/krasnoyarsk

 77%|█████████████████████████████▏        | 972/1268 [00:00<00:00, 1345.40it/s]

https://auto.drom.ru/krasnoyarsk/lexus/rx270/508565308.html
https://auto.drom.ru/belgorod/infiniti/qx70/184042736.html
https://auto.drom.ru/moscow/zeekr/7x/333832308.html
https://auto.drom.ru/moscow/zeekr/001/410838581.html
https://auto.drom.ru/spb/lexus/nx200/930344424.html
https://auto.drom.ru/saratov/zeekr/9x/152086488.html
https://auto.drom.ru/moscow/zeekr/9x/635798018.html
https://auto.drom.ru/novosibirsk/infiniti/m37/856211276.html
https://auto.drom.ru/novosibirsk/lexus/rx300/613282992.html
https://auto.drom.ru/moscow/zeekr/9x/550547886.html
https://auto.drom.ru/krasnodar/zeekr/9x/753956459.html
https://auto.drom.ru/omsk/infiniti/qx50/791288877.html
https://auto.drom.ru/isilkul/lexus/rx350/487318390.html
https://auto.drom.ru/moscow/infiniti/qx70/437211638.html
https://auto.drom.ru/surgut/infiniti/fx37/508406669.html
https://auto.drom.ru/penza/zeekr/9x/500478726.html
https://auto.drom.ru/naberezhnye-chelny/zeekr/001/775553018.html
https://auto.drom.ru/samara/lexus/rx300/566688007.

100%|█████████████████████████████████████| 1268/1268 [00:00<00:00, 1339.71it/s]

https://auto.drom.ru/omsk/geely/tugella/298271628.html
https://auto.drom.ru/omsk/geely/sx11/845583242.html
https://auto.drom.ru/spb/geely/sx11/237433705.html
https://auto.drom.ru/volgodonsk/geely/ex7/804052394.html
https://auto.drom.ru/chelyabinsk/geely/emgrand_iv/724730910.html
https://auto.drom.ru/krasnoyarsk/geely/monjaro/545222316.html
https://auto.drom.ru/krasnoyarsk/geely/ex5_em-i/737626467.html
https://auto.drom.ru/polevskoy/geely/mk_cross/243800446.html
https://auto.drom.ru/ufa/geely/cityray/771845063.html
https://auto.drom.ru/nizhniy-novgorod/geely/okavango/949197353.html
https://auto.drom.ru/ekaterinburg/geely/emgrand_iv/683156657.html
https://auto.drom.ru/tula/geely/cityray/788433269.html
https://auto.drom.ru/moscow/geely/sx11/642935547.html
https://auto.drom.ru/novosibirsk/geely/okavango/451864595.html
https://auto.drom.ru/velikiy-novgorod/geely/tugella/780042918.html
https://auto.drom.ru/moscow/geely/emgrand_iv/756989832.html
https://auto.drom.ru/novosibirsk/geely/cityray/

In [39]:
len(links_2_5_cars)

5066

In [41]:
part1 = pd.read_csv('first_half_checkpoint_4400.csv')
part2 = pd.read_csv('second_half_links_1.csv')
part3 = pd.read_csv('second_half_links_2_check_1250.csv')

In [43]:
part3.iloc[800]

url             https://auto.drom.ru/spb/jaguar/xe/182147165.html
title           Ð¡ Ð²Ð°ÑÐµÐ³Ð¾ IP-Ð°Ð´ÑÐµÑÐ°ÑÐ¾Ð²ÐµÑÑÐµÐ...
year                                                          NaN
city                                                          spb
make                                                       jaguar
model                                                          xe
price                                                         NaN
engine                                                        NaN
transmission                                                  NaN
mileage                                                       NaN
Name: 800, dtype: object

In [45]:
len(part1), len(part2), len(part3)

(4398, 1265, 1250)

In [47]:
part1 = part1.dropna(subset = ['year', 'price', 'engine', 'transmission', 'mileage'], how = 'all')
part2 = part2.dropna(subset = ['year', 'price', 'engine', 'transmission', 'mileage'], how = 'all')
part3 = part3.dropna(subset = ['year', 'price', 'engine', 'transmission', 'mileage'], how = 'all')

In [49]:
len(part1), len(part2), len(part3)

(4364, 1180, 257)

In [51]:
part_1_3 = pd.concat([part1, part2, part3], axis = 0, ignore_index = True)

In [53]:
len(part_1_3)

5801

In [55]:
part1['make'].unique()

array(['toyota', 'hyundai', 'kia', 'honda', 'nissan', 'volkswagen',
       'ford', 'mazda', 'bmw', 'mitsubishi', 'renault', 'lada', 'chery',
       'haval', 'skoda'], dtype=object)

In [57]:
part2['make'].unique()

array(['mercedes-benz', 'audi', 'peugeot', 'citroen', 'opel', 'fiat'],
      dtype=object)

In [59]:
part3['make'].unique()

array(['opel', 'fiat', 'citroen', 'jaguar'], dtype=object)

In [61]:
part_1_3.duplicated(subset = ['url']).sum()

0

In [63]:
part_1_3_list = part_1_3['url'].tolist()

In [133]:
part_1_3.to_csv(f'first_half_total.csv', index=False)

In [135]:
part_1_3['url'].nunique()

5801

In [101]:
url_half_1 = pd.read_csv('all_links.csv')['url'].tolist()
url_half_2 = pd.read_csv('all_links_6_13.csv')['url'].tolist()

In [103]:
len(url_half_1), len(url_half_2)

(4584, 6332)

In [105]:
total_url = url_half_1 + url_half_2

In [107]:
len(total_url)

10916

In [109]:
rest_url = list(set(total_url) - set(part_1_3_list))
len(rest_url)

5115

In [113]:
pd.DataFrame(rest_url).to_csv(f'rest_url.csv', index=False)

In [115]:
rest_url_p_1 = rest_url[:1023]
rest_url_p_2 = rest_url[1023:2046]
rest_url_p_3 = rest_url[2046:3069]
rest_url_p_4 = rest_url[3069:4092]
rest_url_p_5 = rest_url[4092:5115]

In [117]:
pd.DataFrame(rest_url_p_1).to_csv(f'rest_url_p_1.csv', index=False)
pd.DataFrame(rest_url_p_2).to_csv(f'rest_url_p_2.csv', index=False)
pd.DataFrame(rest_url_p_3).to_csv(f'rest_url_p_3.csv', index=False)
pd.DataFrame(rest_url_p_4).to_csv(f'rest_url_p_4.csv', index=False)
pd.DataFrame(rest_url_p_5).to_csv(f'rest_url_p_5.csv', index=False)

In [119]:
url_p1_final = pd.read_csv('url_part_1_final.csv')

In [121]:
len(url_p1_final)

946

In [137]:
url_p2_final = pd.read_csv('url_part_2_final.csv')
len(url_p2_final)

947

In [125]:
url_p3_final = pd.read_csv('url_part_3_final.csv')
len(url_p3_final)

925

In [127]:
url_p5_final = pd.read_csv('url_part_5_final.csv')
len(url_p5_final)

952

In [ ]:
url_p1_final = pd.read_csv('url_part_1_final.csv')
url_p2_final = pd.read_csv('url_part_2_final.csv')
url_p3_final = pd.read_csv('url_part_3_final.csv')
url_p4_final = pd.read_csv('url_part_4_final.csv')
url_p5_final = pd.read_csv('url_part_5_final.csv')

In [ ]:
url_final = pd.concat([url_p1_final, url_p2_final, url_p3_final, url_p4_final, url_p5_final], ignore_index=True)
pd.DataFrame(url_final).to_csv(f'url_final.csv', index=False)